# Rainfall Analysis and Planting Advisory

## Project Overview
This project analyses monthly rainfall data from five Kenyan counties
using Python and NumPy. It identifies annual rainfall totals, wettest
months, dry months and recommended planting windows.

The results will help a farmer-support NGO provide practical
planting advice to farmers through SMS.

In [ ]:
import numpy as np
import csv

In [ ]:
counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

months = [
    "January", "February", "March", "April",
    "May", "June", "July", "August",
    "September", "October", "November", "December"
]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70]
]

machakos_raw = [
    "48", "35", "88", "N/A", "140", " 20 ",
    "9", "12", "", "45", "182", "96"
]


In [ ]:
def clean_machakos(raw_values, month_names):

    valid_values = []

    # Identify valid rainfall values
    for value in raw_values:
        try:
            valid_values.append(int(value.strip()))
        except ValueError:
            continue

    # Calculate the average of valid values
    average_rainfall = round(np.mean(valid_values))

    cleaned_values = []
    imputed_months = []

    # Replace invalid values with the average
    for month, value in zip(month_names, raw_values):
        try:
            cleaned_values.append(int(value.strip()))
        except ValueError:
            cleaned_values.append(average_rainfall)
            imputed_months.append(month)

    print("Average rainfall used for imputation:", average_rainfall)
    print("Imputed months:", imputed_months)

    return cleaned_values


machakos_clean = clean_machakos(machakos_raw, months)

Average rainfall used for imputation: 68
Imputed months: ['April', 'September']


In [ ]:
all_rows = clean_rows + [machakos_clean]

R = np.array(all_rows)

print("Combined rainfall array:")
print(R)

print("Array shape:", R.shape)
print("Array data type:", R.dtype)

Combined rainfall array:
[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Array shape: (5, 12)
Array data type: int64


In [ ]:
annual_totals = np.sum(R, axis=1)

print("Annual rainfall totals:")

for county, total in zip(counties, annual_totals):
    print(county, ":", total, "mm")

Annual rainfall totals:
Nairobi : 1045 mm
Nakuru : 916 mm
Kisumu : 1371 mm
Mombasa : 1075 mm
Machakos : 811 mm


In [ ]:
monthly_averages = np.mean(R, axis=0)

print("Average rainfall per month across all counties:")

for month, average in zip(months, monthly_averages):
    print(month, ":", round(average, 2), "mm")

Average rainfall per month across all counties:
January : 47.0 mm
February : 47.6 mm
March : 91.0 mm
April : 164.4 mm
May : 171.2 mm
June : 67.6 mm
July : 56.6 mm
August : 58.2 mm
September : 64.2 mm
October : 67.8 mm
November : 127.4 mm
December : 80.6 mm


In [ ]:
wettest_indices = np.argmax(R, axis=1)

wettest_months = [months[i] for i in wettest_indices]

print("Wettest month for each county:")

for county, wettest in zip(counties, wettest_months):
    print(county, ":", wettest)

Wettest month for each county:
Nairobi : April
Nakuru : April
Kisumu : April
Mombasa : May
Machakos : November


In [ ]:
dry_mask = R < 50

dry_month_counts = np.sum(dry_mask, axis=1)

print("Number of dry months per county:")

for county, count in zip(counties, dry_month_counts):
    print(county, ":", count)

Number of dry months per county:
Nairobi : 4
Nakuru : 3
Kisumu : 0
Mombasa : 2
Machakos : 6


In [ ]:
# Add rainfall for consecutive months
consecutive_rainfall = R[:, :-1] + R[:, 1:]

# Find the index of the highest combined rainfall
best_window_indices = np.argmax(consecutive_rainfall, axis=1)

# Get the recommended planting windows
planting_windows = [
    f"{months[i]} - {months[i+1]}"
    for i in best_window_indices
]

print("Recommended planting windows:")

for county, window in zip(counties, planting_windows):
    print(county, ":", window)

Recommended planting windows:
Nairobi : April - May
Nakuru : April - May
Kisumu : April - May
Mombasa : April - May
Machakos : November - December


In [ ]:
with open("rainfall_summary.csv", "w", newline="") as file:

    writer = csv.writer(file)

    writer.writerow([
        "county",
        "annual_total_mm",
        "wettest_month",
        "dry_months",
        "planting_window"
    ])

    for row in zip(
        counties,
        annual_totals,
        wettest_months,
        dry_month_counts,
        planting_windows
    ):
        writer.writerow(row)

print("rainfall_summary.csv created successfully!")

rainfall_summary.csv created successfully!


## Planting Advisory

Based on the rainfall analysis, farmers in Nairobi, Nakuru, Kisumu
and Mombasa should consider April to May as their recommended
planting window. Farmers in Machakos should consider November to
December because these months have the highest combined rainfall.
Kisumu recorded the highest annual rainfall at 1,371 mm, while
Machakos recorded the lowest at 811 mm. Farmers should consider
soil moisture, local weather forecasts and crop requirements before
planting. The Machakos data contained two invalid readings that were
replaced with the average of valid readings, so its results should
be interpreted with this data-quality limitation in mind.

## Data Quality Notes

- Machakos rainfall data was initially provided as raw text.
- The values "N/A" and an empty string were invalid.
- April and September were imputed using the rounded average
  of the valid Machakos rainfall readings (68 mm).
- The other four counties were provided as numerical data and
  were used without additional cleaning.
- Imputed values may affect the accuracy of Machakos rainfall
  totals and planting recommendations.